In [ ]:
import pandas as pd  # ler e mexer em tabelas (DataFrames)
import numpy as np # contas matemáticas rápidas com listas de números
import matplotlib.pyplot as plt  # cria gráficos
import seaborn as sns # estilização dos gráficos, e faz mapa de calor (heatmap)
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV  # train_test_split: divide dados em treino/teste; cross_val_score: testa o modelo em várias partes; StratifiedKKFold: garante que cada fold dividido tenha a messma proporção
from sklearn.preprocessing import StandardScaler, OneHotEncoder  # StandardScaler: padroniza números; OneHotEncoder: transforma texto em números
from sklearn.compose import ColumnTransformer  # ColumnTransformer: aplica transformação diferente em cada tipo de coluna
from sklearn.pipeline import Pipeline # Pipeline: junta transformação + modelo numa sequência só (evita vazamento de dados)
from sklearn.svm import SVC  # SVC: o modelo SVM para classificação (Support Vector Classifier)
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay  # métricas (para avaliação do modelo)
from pathlib import Path
import time

plt.rcParams["figure.figsize"] = (8, 5)  # deixa todos os gráficos com proporção padrão 8x5
sns.set_theme(style="whitegrid")  # deixa o fundo dos gráficos com grade branca para melhor visualização
print("Bibliotecas importadas com sucesso!")  # informa o sucesso da operação
print("Versão do Pandas:", pd.__version__)
print("Versão do Numpy:", np.__version__)

In [ ]:
candidates = [Path("/kaggle/input/what-makes-a-yt-video-go-viral-12000-vids/viral_videos.csv"), Path("viral_videos.csv"), Path("../viral_videos.csv")]
local_file = next((file for file in candidates if file.exists()), None)

if local_file is None:
    import kagglehub
    path = Path(kagglehub.dataset_download("vedauv/what-makes-a-yt-video-go-viral-12000-vids"))
    candidates = sorted(path.rglob("*.csv"))
    if not candidates:
        raise FileNotFoundError("CSV não encontrado no download do Kaggle.")
    local_file = next((file for file in candidates if file.name.lower() == "viral_videos.csv"), candidates[0])

print("Arquivo:", local_file)
df = pd.read_csv(local_file, dtype={
    "video_id": str,
    "category": str,
    "duration_sec": np.uint16,
    "title_length": np.uint8,
    "thumbnail_face": bool,
    "upload_hour": np.uint8,
    "upload_day": str,
    "tags_count": np.uint8,
    "description_length": np.uint16,
    "subscriber_count": np.uint32,
    "views": np.uint32,
    "likes": np.uint32,
    "comments": np.uint32,
    "shares": np.uint32,
    "viral": bool
})

# Números inteiros são int64 por padrão até mesmo as colunas com apenas 0 e 1
# Definindo a tipagem dos valores respeitando os limites de memória diminui o uso de memória do DataFrame de 1.4 MB para 621.2 KB
# Ainda mais memória pode ser poupada eliminando a coluna video_id, que não será útil para o treinamento, reduzindo o uso para 527.5 KB
df = df.drop(columns=["video_id"])
df.info()

In [ ]:
random_state = 42
split_samples_by = 2
target_column = "viral"
assert target_column in df.columns, f"Coluna-alvo não encontrada no dataset: \"{target_column}\" em {list(df.columns)}"

df = df.sample(n=len(df.values)//split_samples_by, random_state=random_state, replace=False)

x = df.drop(columns=[target_column])
y = df[target_column]
assert x.notna().all().all() and y.notna().all(), f"Existem valores ausentes nas amostras: \nX:\n{x.values[x.notna()]}\n\nY:\n{y[[not boolean for boolean in y.notna()]]}"

print("Dimensões:", x.shape, "| Classes:", y.nunique())
print("Uso de memória das amostras cruas:", df.memory_usage().sum() / 1000, "KB")
display((df.memory_usage() / 1000).rename("KB").to_frame().transpose())

y_proportions = y.value_counts().sort_index().rename('count').to_frame()
y_proportions["proportions"] = (y.value_counts(normalize=True) * 100).to_frame()

display(
    x.head(),
    y_proportions.transpose(),
    df["category"].value_counts().to_frame().transpose(),
    df["upload_day"].value_counts().to_frame().transpose()
)

In [ ]:
str_columns = x.select_dtypes(include=["object", "string"])

for column in str_columns.columns:
    prepared_encoder = OneHotEncoder(handle_unknown="ignore",categories=[str_columns[column].unique()])
    df[column] = prepared_encoder.fit_transform(pd.DataFrame(str_columns[column])).indices

correlation = df.corr()
plt.figure(figsize=(12, 9))
sns.heatmap(correlation, cmap="coolwarm", center=0)
plt.title("Mapa de calor das correlações")
plt.show()

In [ ]:
irrelevant_cols = set()
correlation_threshold = 0.2

for column_name in correlation.drop(columns=[target_column]).columns:
    column = correlation[column_name].drop(column_name)
    direct_correlation = column[column >= correlation_threshold]
    inverse_correlation = column[column <= -correlation_threshold]
    if len(direct_correlation) + len(inverse_correlation) == 0: irrelevant_cols.add(column_name)
print("Colunas irrelevantes:", irrelevant_cols)

x = x.drop(columns=irrelevant_cols)

correlation = df.drop(columns=irrelevant_cols).corr()
plt.figure(figsize=(6, 5))
sns.heatmap(correlation, annot=False, cmap="coolwarm", center=0)
plt.title("Mapa de calor das correlações (filtradas com as mais relevantes)")

display(
    x.head(),
    f"Uso de memória das amostras filtradas: {(x.memory_usage() / 1000).sum()} KB",
)

In [ ]:
test_size = 0.2

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=test_size, random_state=random_state, stratify=y)

print("Treino:", x_train.shape, f"Uso de memória: {(x_train.memory_usage() / 1000).sum():.2f} KB")
print("Teste:", x_test.shape, f"Uso de memória: {x_test.memory_usage().sum() / 1000:.2f} KB")

print(f"\nImpacto de memória definitivo: {(x_train.memory_usage().sum() + x_test.memory_usage().sum()) / 1000:.2f} KB")

In [ ]:
splits = 5
n_jobs = -1

params_dict = {
    'linear': {'svm__C': [100, 700, 1000, 1500], 'svm__class_weight': [None, 'balanced']},
    'rbf': {'svm__C': [10, 100], 'svm__gamma': ['auto', 'scale', 0.01, 0.1, 1, 10], 'svm__class_weight': [None]},
    'poly': {'svm__C': [1, 10], 'svm__gamma': [1, 10], 'svm__coef0': [0, 1], 'svm__class_weight': [None, 'balanced']},
    'sigmoid': {'svm__C': [1, 10], 'svm__gamma': ['auto', 'scale', 0.001, 0.01, 0.1], 'svm__coef0': [0, 1], 'svm__class_weight': [None, 'balanced']},
}

preprocessing = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(copy=False), list(x.select_dtypes(include="number").columns)),
        # ('cat', OneHotEncoder(handle_unknown='ignore'), list(str_columns.columns))
    ]
)

cv = StratifiedKFold(n_splits=splits, shuffle=True, random_state=random_state)
results, predictions, fitted, cv_tables = [], {}, {}, {}

for kernel, param_grid in params_dict.items():
    print(f'Executando {kernel.upper()}...')
    svm = SVC(kernel=kernel, cache_size=1000, random_state=random_state)
    
    pipeline = Pipeline([
        ('prep', preprocessing),
        ('svm', svm),
    ])
    search = GridSearchCV(pipeline, param_grid=param_grid, cv=cv, scoring=['accuracy', 'f1_macro', 'f1'], n_jobs=n_jobs, refit='accuracy')
    
    start = time.monotonic()
    search.fit(x_train, y_train)
    elapsed = time.monotonic() - start    
    print(f'Modelo treinado em {elapsed:.2f}s')
    
    y_pred = search.predict(x_test)

    fitted[kernel] = search
    predictions[kernel] = y_pred
    cv_tables[kernel] = pd.DataFrame(search.cv_results_).sort_values(by=['rank_test_accuracy', 'rank_test_f1_macro', 'rank_test_f1']).drop(columns=['params', 'std_test_accuracy', 'std_test_f1', 'std_test_f1_macro', 'std_fit_time', 'std_score_time', 'mean_fit_time', 'mean_score_time'])
    results.append({
        'kernel': kernel,
        'cv_best_score': float(search.best_score_),
        'cv_accuracy': float(cv_tables[kernel]['mean_test_accuracy'][0]),
        'cv_f1_binary': float(cv_tables[kernel]['mean_test_f1'][0]),
        'cv_f1_macro': float(cv_tables[kernel]['mean_test_f1_macro'][0]),
        # 'test_accuracy': float(accuracy_score(y_test, y_pred)),
        # 'test_f1_binary': float(f1_score(y_test, y_pred, average='binary')),
        # 'test_f1_macro': float(f1_score(y_test, y_pred, average='macro')),
    })
    
    
    display(
      cv_tables[kernel][:5].transpose(),
      search.best_estimator_,
    )

In [ ]:
results_df = pd.DataFrame(results)

plot_data = results_df.set_index('kernel')[results_df.columns[1:]].sort_values(by=['cv_best_score', 'cv_accuracy', 'cv_f1_binary', 'cv_f1_macro'])[-3:]

ax = plot_data.plot(kind='barh', figsize=(12,6), width=0.6)

ax.set(
  title="Melhor resultado dos 3 melhores kernels",
  xlabel="Score",
  ylabel="Kernel",
  xlim=(plot_data.min().min() - 0.0025, 1.0025),
)
ax.legend(loc='upper left', ncols=1, title='Metric')

for bars in ax.containers:
    ax.bar_label(bars, fmt='%.4f', padding=3)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8, 6))

for (y_idx, x_idx), (svm_type, prediction) in zip(np.ndindex(axes.shape), predictions.items()):
    ax = axes[y_idx, x_idx]
    cm_disp = ConfusionMatrixDisplay(confusion_matrix=confusion_matrix(y_test, prediction), display_labels=['not viral', 'viral'])
    cm_disp.plot(ax=ax)
    cm_disp.ax_.set_title(f'Resultados para o kernel {svm_type.upper()}:')
    ax.set_xlabel("")
    ax.set_ylabel("")

fig.supxlabel(f"Predicted label ({len(x_test)} tests/{len(x)} total)", size=10)
fig.supylabel(f"True label ({len(x_test)}tests/{len(x)} total)", size=10)
plt.tight_layout()
plt.show()

In [ ]:
numeric_features = x_train.select_dtypes(include="number").columns.tolist()
if len(numeric_features) < 2:
    raise ValueError("São necessárias pelo menos duas colunas numéricas para visualizar as fronteiras.")

feature_x, feature_y = numeric_features[:2]
resolution = 180
x_min, x_max = x_train[feature_x].quantile([0.01, 0.99])
y_min, y_max = x_train[feature_y].quantile([0.01, 0.99])
xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, resolution),
    np.linspace(y_min, y_max, resolution),
)

# Mantém as demais variáveis nos valores da primeira amostra de treino.
grid = pd.concat([x_train.iloc[[0]]] * (resolution ** 2), ignore_index=True)
grid[feature_x] = xx.ravel()
grid[feature_y] = yy.ravel()
visible = (
    x_test[feature_x].between(x_min, x_max)
    & x_test[feature_y].between(y_min, y_max)
)

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True, sharey=True)
for ax, (kernel, model) in zip(axes.flat, fitted.items()):
    regions = model.predict(grid).astype(int).reshape(xx.shape)
    ax.contourf(xx, yy, regions, levels=[-0.5, 0.5, 1.5], cmap=plt.cm.RdYlBu, alpha=0.35)

    decision_scores = model.decision_function(grid).reshape(xx.shape)
    ax.contour(xx, yy, decision_scores, levels=[0], colors="black", linewidths=1.5)
    ax.scatter(
        x_test.loc[visible, feature_x], x_test.loc[visible, feature_y],
        c=y_test.loc[visible].astype(int), cmap=plt.cm.RdYlBu,
        s=18, alpha=0.65, edgecolors="white", linewidths=0.25,
    )
    ax.set_title(f"Kernel {kernel.upper()}")
    ax.set_xlabel(feature_x)
    ax.set_ylabel(feature_y)

fig.suptitle(
    f"Fronteiras de decisão por kernel ({feature_x} × {feature_y})\n"
    "Demais variáveis fixadas na primeira amostra de treino; eixos entre os percentis 1 e 99"
)
plt.tight_layout()
plt.show()

In [ ]:
for svm_type, table in cv_tables.items():
    print(f'As piores performances para o SVM {svm_type.upper()}:')
    display(table.sort_values(by=['rank_test_accuracy', 'rank_test_f1_macro', 'rank_test_f1'],ascending=False).head(7).transpose())